# Indonesian Brand (KlikDokter.com) Acquisition

## Check Python Version

In [1]:
!python --version

Python 3.11.9


## Setup Config

In [2]:
DATASET_DIR = "../../datasets/"

import pathlib
OUTPUT_DIR = pathlib.Path(DATASET_DIR)
print(f"Dataset Directory: {OUTPUT_DIR}")

FILE_KLIKDOKTER_DRUG_BRAND_LINKS = "klikdokter_drug_brand_links.csv"
KLIKDOKTER_DRUG_BRAND_LINK_PATH = OUTPUT_DIR.joinpath(FILE_KLIKDOKTER_DRUG_BRAND_LINKS)
print(f"KlikDokter Resolved Drug Brand Links Location: {KLIKDOKTER_DRUG_BRAND_LINK_PATH}")

FILE_DRUG_BRANDS_COMPOSITIONS = "klikdokter_brands.csv"
KLIKDOKTER_DRUG_BRAND_PATH = OUTPUT_DIR.joinpath(FILE_DRUG_BRANDS_COMPOSITIONS)
print(f"KlikDokter Resolved Drug Brands Location: {KLIKDOKTER_DRUG_BRAND_PATH}")

FILE_UNRESOLVED_DRUG_BRANDS_COMPOSITIONS = "klikdokter_unresolved_brands.csv"
KLIKDOKTER_UNRESOLVED_DRUG_BRAND_PATH = OUTPUT_DIR.joinpath(FILE_UNRESOLVED_DRUG_BRANDS_COMPOSITIONS)
print(f"KlikDokter Unresolved Drug Brands and Compositions Location: {KLIKDOKTER_UNRESOLVED_DRUG_BRAND_PATH}")

Dataset Directory: ..\..\datasets
KlikDokter Resolved Drug Brand Links Location: ..\..\datasets\klikdokter_drug_brand_links.csv
KlikDokter Resolved Drug Brands Location: ..\..\datasets\klikdokter_brands.csv
KlikDokter Unresolved Drug Brands and Compositions Location: ..\..\datasets\klikdokter_unresolved_brands.csv


## Add Scrapper

In [3]:
import requests
import pandas as pd
import time
import re
from typing import List
from tqdm import tqdm
from bs4 import BeautifulSoup

class KlikDokterScrapper:
    BASE_URL = "https://www.klikdokter.com"
    HEADERS = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/137.0.0.0 Safari/537.36"
        ),
        "Accept": (
            "text/html,application/xhtml+xml,"
            "application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8"
        ),
        "Accept-Language": "en-US,en;q=0.9,id;q=0.8",
        "Referer": "https://www.google.com/",
        "Connection": "keep-alive",
        "Upgrade-Insecure-Requests": "1",
    }
    
    def get_brand_links(self) -> List[str]:
        all_links = []
        
        for letter in tqdm("abcdefghijklmnopqrstuvwxyz"):
            for page in range(1, 30):
        
                url = (
                    f"{self.BASE_URL}/obat"
                    f"?alphabet={letter}&page={page}"
                )
        
                try:
                    response = requests.get(url, headers=self.HEADERS, timeout=20)
                    if response.status_code != 200:
                        print(f"Failed get: {url}")
                        continue
        
                    soup = BeautifulSoup(response.text, "html.parser")
                    found_links = []
        
                    for a in soup.find_all("a", href=True):
        
                        href = a["href"]
        
                        # filter detail obat
                        if "/obat/" in href:
        
                            # normalize relative url
                            if href.startswith("/"):
                                href = self.BASE_URL + href
        
                            found_links.append(href)
        
                    # remove duplicates in current page
                    found_links = list(set(found_links))
        
                    # kalau page kosong → stop pagination
                    if len(found_links) == 0:
                        break
        
                    all_links.extend(found_links)
                    time.sleep(1)
        
                except Exception as e:
                    print(f"Error Get: {url}, error={e}")

        # deduplicate global
        all_links = list(set(all_links))
        return all_links

    def fetch_brand(self, link):
        try:
            response = requests.get(link, headers=self.HEADERS, timeout=20)
    
            if response.status_code != 200:
                print(f"Failed: {url}")
                return None
    
            soup = BeautifulSoup(response.text, "html.parser")
            product_name = None
    
            title = soup.find("h1")
            if title:
                product_name = title.get_text(strip=True)
    
           
            composition = None
            raw_ingredients = None
            for tag in soup.find_all(["li", "p"]):
                text = tag.get_text(
                    " ",
                    strip=True
                )
            
                if re.match(
                    r"^Kandungan\s*:",
                    text,
                    re.IGNORECASE
                ):
                    raw_ingredients = text
                    break
    
            ingredients = []
            
            if raw_ingredients is None:
                strong_elements = soup.find_all("strong")
                for element in strong_elements:
                    label = element.get_text(strip=True).lower()
                
                    if "kandungan" in label:
                        raw_ingredients = element.parent.get_text(" ", strip=True)
                        break
                        
            if raw_ingredients is None:
                for tr in soup.find_all("tr"):
                    tds = tr.find_all("td")
                    if len(tds) < 2:
                        continue
                
                    key = tds[0].get_text(" ", strip=True).lower()
                    value = tds[1].get_text(" ", strip=True)
                
                    if key == "kandungan":
                        raw_ingredients = value
                        break
                        
            if raw_ingredients is not None:
                pattern = r'([A-Za-z0-9\-\(\)\s]+?)\s+\d+(?:[\.,]\d+)?\s*(?:mg|g|kg|mcg|μg|IU|ml|%)'
                matches = re.findall(pattern, raw_ingredients, flags=re.IGNORECASE)
    
                if len(matches) > 0:
                    for m in matches:
                        cleaned = m.strip()
                        if cleaned.startswith("Kandungan:"):
                            cleaned = cleaned.replace("Kandungan:", "").strip()
                        ingredients.append(cleaned)
                else:
                    cleaned_text = raw_ingredients.replace("Kandungan:", "")
                    split_items = cleaned_text.split(",")
            
                    for item in split_items:
                        cleaned = item.strip()
                        cleaned = re.sub(
                            r"^dan\s+",
                            "",
                            cleaned,
                            flags=re.IGNORECASE
                        )
                        cleaned = cleaned.strip(". ")
                    
                        if len(cleaned) == 0:
                            continue
            
                        ingredients.append(cleaned)
                        
            return {
                "brand_name": product_name,
                "composition_raw": ingredients,
                "url": link
            }
        except Exception as e:
            print(f"Error {link}: {e}")

## Initiate Scrapper

In [4]:
scrapper = KlikDokterScrapper()

## Get All Drug Brand Links

In [5]:
brand_links = scrapper.get_brand_links()

df = pd.DataFrame({"url": brand_links})
df.to_csv(KLIKDOKTER_DRUG_BRAND_LINK_PATH, index=False)
print(f"\nResolved Brand Links: {len(brand_links)}. Links stored at {KLIKDOKTER_DRUG_BRAND_LINK_PATH}")

100%|██████████| 26/26 [07:58<00:00, 18.40s/it]


Resolved Brand Links: 9009. Links stored at ..\..\datasets\klikdokter_drug_brand_links.csv


## Resolve Brand Composition Data from Links 

In [6]:
import pandas as pd
from tqdm import tqdm

brands = []
unknown_brands = []
for _, row in tqdm(df.iterrows(), total=len(df)):
    if not row["url"].startswith("https://www.klikdokter.com/obat"):
        continue
    drug_brand = scrapper.fetch_brand(row["url"])
    if drug_brand is not None:
        brands.append(drug_brand)
    else:
        unknown_brands.append(row["url"])


brands_df = pd.DataFrame(brands)
brands_df.to_csv(KLIKDOKTER_DRUG_BRAND_PATH, index=False)
print(f"\nResolved Drug Brand: {len(brands_df)}. Drug Brands dataset stored at {KLIKDOKTER_DRUG_BRAND_PATH}")

unknown_brands_df = pd.DataFrame({"url": unknown_brands})
unknown_brands_df.to_csv(KLIKDOKTER_UNRESOLVED_DRUG_BRAND_PATH, index=False)
print(f"\nUnresolved Drug Brand: {len(unknown_brands_df)}. Unresolved Drug Brand Links stored at {KLIKDOKTER_UNRESOLVED_DRUG_BRAND_PATH}")

100%|██████████| 9009/9009 [2:00:08<00:00,  1.25it/s]  


Resolved Drug Brand: 9009. Drug Brands dataset stored at ..\..\datasets\klikdokter_brands.csv

Unresolved Drug Brand: 0. Unresolved Drug Brand Links stored at ..\..\datasets\klikdokter_unresolved_brands.csv


## Check Brand with no ingredients and exist ingredients

In [10]:
import pandas as pd
import ast

brands = pd.read_csv(KLIKDOKTER_DRUG_BRAND_PATH)
brands["composition_raw"] = brands["composition_raw"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)
empty_compositions = brands[
    brands["composition_raw"].apply(
        lambda x: isinstance(x, list) and len(x) == 0
    )
]
exist_compositions = brands[
    brands["composition_raw"].apply(
        lambda x: isinstance(x, list) and len(x) > 0
    )
]

print(f"\nDrug Brand with ingredients: {len(exist_compositions)}.")
print(f"\nDrug Brand with no ingredients: {len(empty_compositions)}.")


Drug Brand with ingredients: 8924.

Drug Brand with no ingredients: 85.


In [11]:
print(exist_compositions.head())
print(empty_compositions.head())

           brand_name                                    composition_raw  \
0            Anhissen                                       [Loratadine]   
1            Neobumin      [Ekstrak Ophiocephalus striatus (ikan gabus)]   
2  Natrol Acidophilus                        [Lactobacillus Acidophilus]   
3             Lanovit  [Vitamin B1, Vitamin B2, Vitamin B6, Vitamin B...   
4        Tetracycline                 [Tetracyline HCl, Tetracyline HCl]   

                                                 url  
0  https://www.klikdokter.com/obat/obat-alergi/an...  
1  https://www.klikdokter.com/obat/vitamin-dan-su...  
2  https://www.klikdokter.com/obat/vitamin-dan-su...  
3  https://www.klikdokter.com/obat/vitamin-dan-su...  
4  https://www.klikdokter.com/obat/obat-antibioti...  
    brand_name composition_raw  \
28    BMT Soya              []   
51     Betaflu              []   
78     Erlagin              []   
215   Burnazin              []   
356    Twynsta              []   

        